# Backtesting Analytics
Compares model predictions against actual price direction for every day in the held-out test set.

**Data source:** `reports/backtesting/combined_detail.csv`  
**Sections:**
1. Setup and data load
2. Single ticker analysis — tables and charts
3. All tickers combined — tables and charts


## 1. Setup

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display

# ── Load data ──────────────────────────────────────────────────────────────
DATA_PATH = "reports/backtesting/combined_detail.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["date"])
df["month"]  = df["date"].dt.to_period("M").astype(str)
df["correct"] = df["correct"].astype(bool)

MODELS  = sorted(df["model"].unique())
TICKERS = sorted(df["ticker"].unique())

MODEL_COLORS = {
    "gradient_boosting":   "#BA7517",
    "logistic_regression": "#1D9E75",
    "random_forest":       "#534AB7",
}

MODEL_LABELS = {
    "gradient_boosting":   "Gradient Boosting",
    "logistic_regression": "Logistic Regression",
    "random_forest":       "Random Forest",
}

print(f"Loaded {len(df):,} rows")
print(f"Tickers : {TICKERS}")
print(f"Models  : {MODELS}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")

Loaded 12,480 rows
Tickers : ['AAPL', 'AMC', 'AMZN', 'GME', 'GOOGL', 'MSFT', 'NFLX', 'SPY', 'SYY', 'USFD']
Models  : ['gradient_boosting', 'logistic_regression', 'random_forest']
Date range: 2024-10-17 to 2026-06-16


---
## 2. Single Ticker Analysis
Set `TICKER` below to the symbol you want to explore.


In [2]:
TICKER = TICKERS[0]   # change to any ticker in your dataset e.g. "MSFT"

t = df[df["ticker"] == TICKER].copy()
print(f"Analysing {TICKER}  —  {len(t):,} rows  "
      f"({t['date'].min().date()} to {t['date'].max().date()})")

Analysing AAPL  —  1,248 rows  (2024-10-17 to 2026-06-16)


### 2a. Overall Accuracy by Model

In [3]:
summary = (
    t.groupby("model")
    .agg(
        Total        = ("correct", "count"),
        Correct      = ("correct", "sum"),
        Predicted_Up = ("predicted", lambda x: (x == 1).sum()),
        Predicted_Dn = ("predicted", lambda x: (x == 0).sum()),
        Actual_Up    = ("actual",    lambda x: (x == 1).sum()),
        Actual_Dn    = ("actual",    lambda x: (x == 0).sum()),
    )
    .assign(
        Accuracy  = lambda x: (x["Correct"] / x["Total"]).map("{:.1%}".format),
        Bias      = lambda x: (x["Predicted_Up"] / x["Total"]).map("{:.1%}".format),
    )
    .rename(index=MODEL_LABELS)
    .reset_index()
    .rename(columns={"model": "Model"})
)

display(summary[["Model","Total","Correct","Accuracy","Predicted_Up","Predicted_Dn","Bias"]])

,Model,Total,Correct,Accuracy,Predicted_Up,Predicted_Dn,Bias
0,Gradient Boosting,416,236,56.7%,338,78,81.2%
1,Logistic Regression,416,212,51.0%,392,24,94.2%
2,Random Forest,416,226,54.3%,370,46,88.9%


### 2b. Monthly Accuracy by Model

In [8]:
monthly = (
    t.groupby(["month", "model"])["correct"]
    .mean()
    .mul(100)
    .round(1)
    .unstack("model")
    .rename(columns=MODEL_LABELS)
    .reset_index()
    .rename(columns={"month": "Month"})
)

# Highlight cells above 55% and below 45%
def highlight(val):
    if not isinstance(val, float):
        return ""
    if val >= 55:
        return "background-color: #d4edda; color: #155724"
    if val <= 45:
        return "background-color: #f8d7da; color: #721c24"
    return ""

display(
    monthly.style
    # .applymap(highlight, subset=[MODEL_LABELS[m] for m in MODELS if MODEL_LABELS[m] in monthly.columns])
    .map(highlight, subset=[MODEL_LABELS[m] for m in MODELS if MODEL_LABELS[m] in monthly.columns])
    .format({MODEL_LABELS[m]: "{:.1f}%" for m in MODELS if MODEL_LABELS[m] in monthly.columns})
    .set_caption(f"{TICKER} — Monthly accuracy (green ≥ 55%, red ≤ 45%)")
)

model,Month,Gradient Boosting,Logistic Regression,Random Forest
0,2024-10,45.5%,45.5%,45.5%
1,2024-11,75.0%,60.0%,70.0%
2,2024-12,52.4%,66.7%,38.1%
3,2025-01,35.0%,30.0%,35.0%
4,2025-02,63.2%,52.6%,57.9%
5,2025-03,52.4%,42.9%,47.6%
6,2025-04,66.7%,47.6%,57.1%
7,2025-05,38.1%,33.3%,38.1%
8,2025-06,55.0%,60.0%,60.0%
9,2025-07,50.0%,54.5%,59.1%


### 2c. Predictions vs Actuals Over Time

In [9]:
model_sel = t["model"].iloc[0]   # change to compare a different model

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.6, 0.4],
    vertical_spacing=0.06,
    subplot_titles=[
        f"{TICKER} — Predicted vs Actual Direction ({MODEL_LABELS[model_sel]})",
        "Close Price"
    ]
)

sub = t[t["model"] == model_sel].sort_values("date")

# Correct calls
correct = sub[sub["correct"] == True]
wrong   = sub[sub["correct"] == False]

fig.add_trace(go.Scatter(
    x=correct["date"], y=correct["predicted_label"],
    mode="markers", name="Correct",
    marker=dict(color="#1D9E75", size=6, symbol="circle"),
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=wrong["date"], y=wrong["predicted_label"],
    mode="markers", name="Wrong",
    marker=dict(color="#D85A30", size=6, symbol="x"),
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=sub["date"], y=sub["actual_label"],
    mode="lines", name="Actual",
    line=dict(color="#888780", width=1, dash="dot"),
), row=1, col=1)

# Close price
fig.add_trace(go.Scatter(
    x=sub["date"], y=sub["Close"],
    mode="lines", name="Close",
    line=dict(color="#378ADD", width=1.5),
    fill="tozeroy", fillcolor="rgba(55,122,221,0.08)"
), row=2, col=1)

fig.update_layout(height=520, hovermode="x unified", legend=dict(orientation="h", y=1.08))
fig.update_yaxes(title_text="Direction", row=1, col=1)
fig.update_yaxes(title_text="Price ($)",  row=2, col=1)
fig.show()

### 2d. Monthly Accuracy by Model

In [10]:
monthly_long = (
    t.groupby(["month", "model"])["correct"]
    .mean()
    .mul(100)
    .round(1)
    .reset_index()
    .rename(columns={"correct": "accuracy", "model": "Model"})
)
monthly_long["Model"] = monthly_long["Model"].map(MODEL_LABELS)

fig = go.Figure()

for model_name in [MODEL_LABELS[m] for m in MODELS]:
    sub = monthly_long[monthly_long["Model"] == model_name]
    fig.add_trace(go.Scatter(
        x=sub["month"], y=sub["accuracy"],
        mode="lines+markers", name=model_name,
        line=dict(color=[v for k,v in MODEL_COLORS.items() if MODEL_LABELS[k] == model_name][0], width=2),
        marker=dict(size=7),
    ))

fig.add_hline(y=50, line_dash="dash", line_color="#888780",
              annotation_text="50% — random chance", annotation_position="top left")
fig.add_hrect(y0=55, y1=100, fillcolor="#1D9E75", opacity=0.04, line_width=0)
fig.add_hrect(y0=0,  y1=45,  fillcolor="#D85A30", opacity=0.04, line_width=0)

fig.update_layout(
    title=f"{TICKER} — Monthly Accuracy by Model",
    xaxis_title="Month", yaxis_title="Accuracy (%)",
    height=400, hovermode="x unified",
    legend=dict(orientation="h", y=1.1),
    yaxis=dict(range=[30, 80])
)
fig.show()

---
## 3. All Tickers Combined


### 3a. Overall Accuracy — All Tickers

In [11]:
all_summary = (
    df.groupby(["ticker", "model"])
    .agg(
        Total   = ("correct", "count"),
        Correct = ("correct", "sum"),
    )
    .assign(Accuracy = lambda x: (x["Correct"] / x["Total"]).map("{:.1%}".format))
    .reset_index()
    .rename(columns={"ticker": "Ticker", "model": "Model"})
)
all_summary["Model"] = all_summary["Model"].map(MODEL_LABELS)

pivot = all_summary.pivot(index="Ticker", columns="Model", values="Accuracy")
display(pivot)

Model,Gradient Boosting,Logistic Regression,Random Forest
Ticker,,,
AAPL,56.7%,51.0%,54.3%
AMC,56.5%,59.6%,59.1%
AMZN,52.4%,54.1%,51.2%
GME,55.3%,52.4%,52.9%
GOOGL,54.6%,56.0%,52.6%
MSFT,49.8%,51.4%,48.6%
NFLX,48.6%,47.1%,46.6%
SPY,53.1%,55.0%,54.1%
SYY,51.7%,49.3%,47.6%


### 3b. Best Model per Ticker

In [12]:
best = (
    df.groupby(["ticker", "model"])
    .agg(Accuracy=("correct", "mean"), Total=("correct", "count"))
    .reset_index()
)
best["Model"] = best["model"].map(MODEL_LABELS)

idx   = best.groupby("ticker")["Accuracy"].idxmax()
table = best.loc[idx, ["ticker","Model","Accuracy","Total"]].copy()
table["Accuracy"] = table["Accuracy"].map("{:.1%}".format)
table = table.rename(columns={"ticker": "Ticker", "Total": "Test Days"})

display(table.reset_index(drop=True))

,Ticker,Model,Accuracy,Test Days
0,AAPL,Gradient Boosting,56.7%,416
1,AMC,Logistic Regression,59.6%,416
2,AMZN,Logistic Regression,54.1%,416
3,GME,Gradient Boosting,55.3%,416
4,GOOGL,Logistic Regression,56.0%,416
5,MSFT,Logistic Regression,51.4%,416
6,NFLX,Gradient Boosting,48.6%,416
7,SPY,Logistic Regression,55.0%,416
8,SYY,Gradient Boosting,51.7%,416
9,USFD,Gradient Boosting,54.8%,416


### 3c. Accuracy by Model — All Tickers

In [13]:
acc_all = (
    df.groupby(["ticker", "model"])["correct"]
    .mean()
    .mul(100)
    .round(1)
    .reset_index()
    .rename(columns={"correct": "Accuracy", "model": "Model", "ticker": "Ticker"})
)
acc_all["Model"] = acc_all["Model"].map(MODEL_LABELS)

fig = px.bar(
    acc_all, x="Ticker", y="Accuracy", color="Model",
    barmode="group",
    color_discrete_map={MODEL_LABELS[k]: v for k,v in MODEL_COLORS.items()},
    title="Overall Accuracy by Model and Ticker",
    height=420
)
fig.add_hline(y=50, line_dash="dash", line_color="#888780",
              annotation_text="50% baseline")
fig.update_layout(yaxis=dict(range=[30, 75]), legend=dict(orientation="h", y=1.1))
fig.show()

### 3d. Monthly Accuracy — All Tickers

In [14]:
monthly_all = (
    df.groupby(["ticker", "month", "model"])["correct"]
    .mean()
    .mul(100)
    .round(1)
    .reset_index()
    .rename(columns={"correct": "Accuracy", "model": "Model", "ticker": "Ticker"})
)
monthly_all["Model"] = monthly_all["Model"].map(MODEL_LABELS)

fig = px.line(
    monthly_all, x="month", y="Accuracy",
    color="Model", facet_row="Ticker",
    color_discrete_map={MODEL_LABELS[k]: v for k,v in MODEL_COLORS.items()},
    title="Monthly Accuracy by Model — All Tickers",
    markers=True, height=300 * len(TICKERS)
)
fig.add_hline(y=50, line_dash="dash", line_color="#888780")
fig.update_layout(hovermode="x unified", legend=dict(orientation="h", y=1.02))
fig.update_xaxes(tickangle=45)
fig.update_yaxes(range=[25, 80])
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()